## 03 — Prepare Real Data (Roboflow)

### 0. Setup

In [3]:
# Core imports
import shutil
from pathlib import Path
from collections import Counter

import yaml

# -- Project root --------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = Path("..") if NOTEBOOK_DIR.name == "notebooks" else Path(".")
print(f"Project root: {PROJECT_ROOT}")

# -- Paths (all under data/) ---------------------------------------------------
DATA_DIR       = PROJECT_ROOT / "data"
FINETUNE_DIR   = DATA_DIR / "noodles_finetune_dataset"

# -- Shared constants (14 classes: 0-10 pieces, 11 board, 12 hinge, 13 pin) ---
PIECE_LABELS    = list("ABCDEFGHIJK")                              # piece classes 0-10
ALL_CLASS_NAMES = PIECE_LABELS + ["board", "hinge", "pin"]         # classes 11, 12, 13
NUM_CLASSES     = len(ALL_CLASS_NAMES)                             # 14
PIN_CLASS_ID    = 13
HINGE_CLASS_ID  = 12
EXPECTED_PINS_PER_IMAGE = 21

# Hinge (class 12) is reserved in the schema but NOT trained on for now.
# Flip SKIP_HINGE to False later when we want to include hinge annotations.
SKIP_HINGE = True

PIECE_COLORS = {
    'A': ('Yellow',      (0xF9, 0xD6, 0x5E)),
    'B': ('SkyBlue',     (0x08, 0xA7, 0xE8)),
    'C': ('DarkBlue',    (0x20, 0x6D, 0xD9)),
    'D': ('Green',       (0x1F, 0xA1, 0x5B)),
    'E': ('Red',         (0xEE, 0x39, 0x4F)),
    'F': ('Teal',        (0x85, 0xDA, 0xBB)),
    'G': ('Pink',        (0xEC, 0x71, 0xA8)),
    'H': ('Purple',      (0xC7, 0x78, 0xB9)),
    'I': ('Orange',      (0xFC, 0x69, 0x0C)),
    'J': ('DarkRed',     (0xB6, 0x30, 0x48)),
    'K': ('YellowGreen', (0x95, 0xD4, 0x50)),
}

# Build unified class mapping (all 14 classes)
UNIFIED_NAMES = {
    i: f"{label}_{PIECE_COLORS[label][0]}"
    for i, label in enumerate(PIECE_LABELS)
}
UNIFIED_NAMES[11] = 'board'
UNIFIED_NAMES[12] = 'hinge'
UNIFIED_NAMES[13] = 'pin'

print(f"\nSetup complete - {NUM_CLASSES} classes")
print(f"   Fine-tune dir: {FINETUNE_DIR}")
if SKIP_HINGE:
    print(f"   Hinge (class 12) will be SKIPPED during prep (not trained on)")


Project root: ..

Setup complete - 14 classes
   Fine-tune dir: ..\data\noodles_finetune_dataset
   Hinge (class 12) will be SKIPPED during prep (not trained on)


### 1. Download Real Data from Roboflow

In [4]:
# !pip install -q roboflow

from roboflow import Roboflow

# ⚠️  FILL IN your Roboflow project details below
ROBOFLOW_API_KEY   = "rtBCMfDQl6zSFnAQzsHn"
ROBOFLOW_WORKSPACE = "abdulsalams-workspace-cslqv"
ROBOFLOW_PROJECT   = "noodels"
ROBOFLOW_VERSION   = 4  # ← bump this to your latest version with board+hinge annotations

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

# Download in YOLOv8 segmentation format
dataset = project.version(ROBOFLOW_VERSION).download("yolov8")

REAL_DATA_DIR = Path(dataset.location)
print(f"\n✅ Real dataset downloaded to: {REAL_DATA_DIR}")
print(f"   Train: {REAL_DATA_DIR / 'train'}")
print(f"   Val:   {REAL_DATA_DIR / 'valid'}")

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 0.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to noodels-4 in yolov8:: 100%|██████████| 209/209 [00:00<00:00, 1084.36it/s]



✅ Real dataset downloaded to: c:\Users\abdul\Desktop\noodels\notebooks\noodels-4
   Train: c:\Users\abdul\Desktop\noodels\notebooks\noodels-4\train
   Val:   c:\Users\abdul\Desktop\noodels\notebooks\noodels-4\valid


### 2. Validate Roboflow Labels

In [5]:
# ── Read Roboflow's data.yaml ─────────────────────────────────────────────────
rf_yaml_path = REAL_DATA_DIR / 'data.yaml'
with open(rf_yaml_path) as f:
    rf_config = yaml.safe_load(f)

rf_names = rf_config.get('names', {})
if isinstance(rf_names, list):
    rf_names = {i: n for i, n in enumerate(rf_names)}

print(f"Unified target mapping ({NUM_CLASSES} classes):")
for k, v in UNIFIED_NAMES.items():
    print(f"  {k}: {v}")

print("\nRoboflow class mapping:")
for k, v in sorted(rf_names.items(), key=lambda x: int(x[0])):
    print(f"  {k}: {v}")

# ── Verify match ──────────────────────────────────────────────────────────────
# All 14 ids (0..13) exist in UNIFIED_NAMES — pieces 0–10, board 11, hinge 12,
# pin 13. We expect pieces to match by letter and 11–13 to match by name.
remap = {}
mismatches = []

for rf_id, rf_name in rf_names.items():
    rf_id = int(rf_id)
    if rf_id not in UNIFIED_NAMES:
        mismatches.append(f"  ID {rf_id}: '{rf_name}' — not in unified scheme (expected 0..{NUM_CLASSES-1})")
        continue

    uni_name = UNIFIED_NAMES[rf_id]
    if rf_id < len(PIECE_LABELS):
        # Piece class — compare by letter (e.g. 'A_Yellow' vs 'A_Yellow').
        rf_letter = rf_name.strip().split('_')[0].upper()
        uni_letter = uni_name.split('_')[0].upper()
        if rf_letter != uni_letter:
            mismatches.append(f"  ID {rf_id}: Roboflow='{rf_name}' vs Unified='{uni_name}'")
    else:
        # board/hinge/pin — compare by exact name (case-insensitive).
        if rf_name.strip().lower() != uni_name.lower():
            mismatches.append(f"  ID {rf_id}: Roboflow='{rf_name}' vs Unified='{uni_name}'")

    remap[rf_id] = rf_id

if mismatches:
    print(f"\n⚠️  {len(mismatches)} potential mismatches:")
    for m in mismatches:
        print(m)
    print("\nProceeding anyway — class IDs will be kept as-is.")
else:
    print(f"\n✅ All {len(remap)} Roboflow classes match unified scheme perfectly!")
    print("   No remapping needed — labels will be copied as-is.")

Unified target mapping (14 classes):
  0: A_Yellow
  1: B_SkyBlue
  2: C_DarkBlue
  3: D_Green
  4: E_Red
  5: F_Teal
  6: G_Pink
  7: H_Purple
  8: I_Orange
  9: J_DarkRed
  10: K_YellowGreen
  11: board
  12: hinge
  13: pin

Roboflow class mapping:
  0: A_Yellow
  1: B_SkyBlue
  2: C_DarkBlue
  3: D_Green
  4: E_Red
  5: F_Teal
  6: G_Pink
  7: H_Purple
  8: I_Orange
  9: J_DarkRed
  10: K_YellowGreen
  11: board
  12: hinge
  13: pin

✅ All 14 Roboflow classes match unified scheme perfectly!
   No remapping needed — labels will be copied as-is.


### 3. Prepare Fine-Tune Dataset

In [6]:
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

stats = {
    'copied': 0,
    'skipped_lines': 0,
    'skipped_hinge': 0,
    'total_files': 0,
    'total_images': 0,
    'images_skipped_partial_pins': 0,
}
# Per-image pin count for the "all 21 or skip" gate.
pins_per_image = {}

for split_name, rf_split in [('train', 'train'), ('val', 'valid'), ('val', 'val')]:
    rf_img_dir = REAL_DATA_DIR / rf_split / 'images'
    rf_lbl_dir = REAL_DATA_DIR / rf_split / 'labels'

    if not rf_img_dir.exists() or not rf_lbl_dir.exists():
        continue

    out_img_dir = FINETUNE_DIR / 'images' / split_name
    out_lbl_dir = FINETUNE_DIR / 'labels' / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    label_files = list(rf_lbl_dir.glob('*.txt'))
    print(f"\n{rf_split} → {split_name}: {len(label_files)} label files")

    for lbl_file in label_files:
        valid_lines = []
        pin_count = 0
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    stats['skipped_lines'] += 1
                    continue
                cls_id = int(parts[0])
                n_vals = len(parts) - 1

                if cls_id < 0 or cls_id >= NUM_CLASSES:
                    stats['skipped_lines'] += 1
                    continue

                # Skip hinge annotations for now — not training on class 12 yet.
                if SKIP_HINGE and cls_id == HINGE_CLASS_ID:
                    stats['skipped_hinge'] += 1
                    continue

                # YOLO-seg format: every class is a polygon (>=6 coords, even count).
                # Roboflow YOLOv8-seg export and the synthetic generator both emit
                # polygons for all classes. Mixing 4-value bboxes with polygons in
                # a seg dataset breaks training.
                if n_vals >= 6 and n_vals % 2 == 0:
                    valid_lines.append(line.strip())
                    stats['copied'] += 1
                    if cls_id == PIN_CLASS_ID:
                        pin_count += 1
                else:
                    stats['skipped_lines'] += 1

        pins_per_image[lbl_file.stem] = pin_count

        # "All 21 pins or skip" gate. Partial pin annotations teach the model
        # that un-annotated pins are background, which tanks recall. 0 pins is
        # fine (image may have the board fully out of frame); 1..20 is dropped.
        if 0 < pin_count < EXPECTED_PINS_PER_IMAGE:
            stats['images_skipped_partial_pins'] += 1
            continue

        out_lbl = out_lbl_dir / lbl_file.name
        with open(out_lbl, 'w') as f:
            f.write('\n'.join(valid_lines) + '\n' if valid_lines else '')
        stats['total_files'] += 1

        img_stem = lbl_file.stem
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            src_img = rf_img_dir / (img_stem + ext)
            if src_img.exists():
                dst_img = out_img_dir / src_img.name
                if not dst_img.exists():
                    shutil.copy2(src_img, dst_img)
                stats['total_images'] += 1
                break

print(f"\n{'='*50}")
print(f"✅ Dataset prepared — {NUM_CLASSES} classes!")
print(f"   Label files: {stats['total_files']}")
print(f"   Images:      {stats['total_images']}")
print(f"   Annotations: {stats['copied']}")
if stats['skipped_hinge']:
    print(f"   ℹ️  Hinge lines skipped (SKIP_HINGE=True): {stats['skipped_hinge']}")
if stats['skipped_lines']:
    print(f"   ⚠️  Skipped lines:                   {stats['skipped_lines']}")
if stats['images_skipped_partial_pins']:
    print(f"   ⚠️  Images dropped (partial pins):   {stats['images_skipped_partial_pins']}")

# Pin-coverage histogram: every image with pins should have exactly 21.
pin_hist = Counter(pins_per_image.values())
print("\nPin-count histogram (how many images have N pins annotated):")
for n in sorted(pin_hist.keys()):
    if n == EXPECTED_PINS_PER_IMAGE:
        flag = '  ← expected'
    elif n == 0:
        flag = '  ← empty (no pins annotated)'
    else:
        flag = '  ⚠️  partial — dropped'
    print(f"  {n:2d} pins: {pin_hist[n]:4d} images{flag}")



train → train: 103 label files

✅ Dataset prepared — 14 classes!
   Label files: 101
   Images:      101
   Annotations: 2566
   ℹ️  Hinge lines skipped (SKIP_HINGE=True): 103
   ⚠️  Images dropped (partial pins):   2

Pin-count histogram (how many images have N pins annotated):
   1 pins:    1 images  ⚠️  partial — dropped
  20 pins:    1 images  ⚠️  partial — dropped
  21 pins:  101 images  ← expected


### 4. Create Fine-Tune Dataset YAML

In [ ]:
finetune_yaml_content = f"""# IQ Noodles Fine-Tune Dataset - Real Photos (Phase 2)
# 14 classes: pieces A-K (0-10), board (11), hinge (12), pin (13)

path: {FINETUNE_DIR.resolve()}
train: images/train
val: images/val

nc: {NUM_CLASSES}

names:
"""
for idx, name in UNIFIED_NAMES.items():
    finetune_yaml_content += f"  {idx}: {name}\n"

finetune_yaml_path = FINETUNE_DIR / 'dataset.yaml'
with open(finetune_yaml_path, 'w') as f:
    f.write(finetune_yaml_content)

print(f"Fine-tune dataset YAML: {finetune_yaml_path}")
print(finetune_yaml_content)


### 5. Validate Labels

In [8]:
def validate_labels(dataset_dir, split='train'):
    """Check label files for correctness (14 classes, all polygon format)."""
    lbl_dir = Path(dataset_dir) / 'labels' / split
    img_dir = Path(dataset_dir) / 'images' / split

    label_files = sorted(lbl_dir.glob('*.txt'))
    image_files = sorted(img_dir.glob('*'))
    image_stems = {f.stem for f in image_files}

    print(f"  Images: {len(image_files)}  |  Labels: {len(label_files)}")

    class_counts = Counter()
    orphan_labels = []
    bad_lines = []
    partial_pin_images = []
    total_annotations = 0

    for lf in label_files:
        if lf.stem not in image_stems:
            orphan_labels.append(lf.name)

        pin_count_this_file = 0
        with open(lf) as f:
            for line_no, line in enumerate(f, 1):
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                n_vals = len(parts) - 1

                if cls_id < 0 or cls_id >= NUM_CLASSES:
                    bad_lines.append(f"{lf.name}:{line_no} class_id={cls_id} out of range [0,{NUM_CLASSES-1}]")
                else:
                    # All classes (pieces 0-10, board 11, hinge 12, pin 13) use
                    # YOLO-seg polygon format: >=6 coords, even count, in [0,1].
                    if n_vals < 6 or n_vals % 2 != 0:
                        bad_lines.append(f"{lf.name}:{line_no} invalid polygon ({n_vals} coords)")
                    else:
                        coords = [float(x) for x in parts[1:]]
                        if any(c < 0 or c > 1 for c in coords):
                            bad_lines.append(f"{lf.name}:{line_no} coords outside [0,1]")
                    if cls_id == 13:
                        pin_count_this_file += 1

                class_counts[cls_id] += 1
                total_annotations += 1

        # Pin-coverage per image: 0 or 21 are acceptable; anything else is a bug.
        if 0 < pin_count_this_file < 21:
            partial_pin_images.append(f"{lf.name} ({pin_count_this_file} pins)")

    print(f"  Total annotations: {total_annotations}")
    print(f"  Class distribution:")
    for cls_id in sorted(class_counts.keys()):
        name = UNIFIED_NAMES.get(cls_id, f"UNKNOWN_{cls_id}")
        print(f"    {cls_id:2d} ({name}): {class_counts[cls_id]}")

    if orphan_labels:
        print(f"\n  ⚠️  {len(orphan_labels)} label files without matching image")
    if partial_pin_images:
        print(f"\n  ❌ {len(partial_pin_images)} images with partial pin annotation (expected 0 or 21):")
        for p in partial_pin_images[:10]:
            print(f"    {p}")
    if bad_lines:
        print(f"\n  ❌ {len(bad_lines)} problematic lines:")
        for bl in bad_lines[:10]:
            print(f"    {bl}")

    all_ok = not bad_lines and not partial_pin_images
    if all_ok:
        print(f"\n  ✅ All labels valid!")
    return all_ok

print("=== Train split ===")
train_ok = validate_labels(FINETUNE_DIR, 'train')
print("\n=== Val split ===")
val_ok = validate_labels(FINETUNE_DIR, 'val')

if train_ok and val_ok:
    print(f"\n✅ All labels validated — {NUM_CLASSES} classes, ready for fine-tuning!")
else:
    print("\n⚠️  Fix label issues above before proceeding")


=== Train split ===
  Images: 101  |  Labels: 101
  Total annotations: 2531
  Class distribution:
     0 (A_Yellow): 23
     1 (B_SkyBlue): 26
     2 (C_DarkBlue): 24
     3 (D_Green): 34
     4 (E_Red): 34
     5 (F_Teal): 19
     6 (G_Pink): 29
     7 (H_Purple): 26
     8 (I_Orange): 24
     9 (J_DarkRed): 36
    10 (K_YellowGreen): 34
    11 (board): 101
    13 (pin): 2121

  ✅ All labels valid!

=== Val split ===
  Images: 0  |  Labels: 0
  Total annotations: 0
  Class distribution:

  ✅ All labels valid!

✅ All labels validated — 14 classes, ready for fine-tuning!
